In [1]:
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from scipy.special import softmax
from sklearn.metrics import roc_auc_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# Load the dataset
data = pd.read_csv("C:/Users/Stiller/Documents/Daniel's Documents/CS747 Final Project/data/CS747data/CS747data.csv", sep=',')

# Remove source column
data = data.iloc[:,:-1]

# Rename label column
data = data.rename(columns={"label": "labels"})

data.head()

,text,labels
0,## The Advantages of Limiting Car Usage\n\nIn ...,1
1,"The United States is a car-dependent nation, w...",1
2,"In recent years, there has been a growing move...",1
3,"In recent years, there has been a growing move...",1
4,"In the past few decades, the United States has...",1


In [4]:
# Split data
train, temp = train_test_split(data, test_size=0.2, random_state=42, stratify=data["labels"]) # 80% train
val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp["labels"]) #10% val, 10% test

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (132068, 2)
Validation: (16509, 2)
Test: (16509, 2)


In [5]:
# Save data for later if needed
train.to_csv("./BertData/BERTtrain.csv", index=False)
val.to_csv("./BertData/BERTval.csv", index=False)
test.to_csv("./BertData/BERTtest.csv", index=False)

In [6]:
# Convert to hugging face dataset
train = Dataset.from_pandas(train)
val = Dataset.from_pandas(val)
test = Dataset.from_pandas(test)

In [7]:
# Tokenize the essays
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train = train.map(tokenize, batched=True)
val = val.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)

Map: 100%|██████████| 16509/16509 [00:08<00:00, 1981.21 examples/s]


In [8]:
# Load BERT model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model.to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 329.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [9]:
# Train the model
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = softmax(logits, axis=1)[:,1]

    auc = roc_auc_score(labels, probs)

    return {
        "auc": auc
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Auc
1,0.136166,0.174846,0.959373
2,0.574864,0.578283,0.503523
3,0.595921,0.579785,0.483570


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=49527, training_loss=0.40108237448602646, metrics={'train_runtime': 48941.7935, 'train_samples_per_second': 8.095, 'train_steps_per_second': 1.012, 'total_flos': 1.0424565257785344e+17, 'train_loss': 0.40108237448602646, 'epoch': 3.0})

In [10]:
trainer.state.log_history

[{'loss': 0.1302608184814453,
  'grad_norm': 20.525070190429688,
  'learning_rate': 4.984887031316252e-05,
  'epoch': 0.030286510388273062,
  'step': 500},
 {'loss': 0.062053504943847655,
  'grad_norm': 0.06125233694911003,
  'learning_rate': 4.9697437761221155e-05,
  'epoch': 0.060573020776546124,
  'step': 1000},
 {'loss': 0.07250719451904297,
  'grad_norm': 0.06980200856924057,
  'learning_rate': 4.954600520927979e-05,
  'epoch': 0.0908595311648192,
  'step': 1500},
 {'loss': 0.06376766204833985,
  'grad_norm': 0.0022333962842822075,
  'learning_rate': 4.939457265733842e-05,
  'epoch': 0.12114604155309225,
  'step': 2000},
 {'loss': 0.057072261810302734,
  'grad_norm': 0.11444654315710068,
  'learning_rate': 4.9243140105397054e-05,
  'epoch': 0.15143255194136532,
  'step': 2500},
 {'loss': 0.05798492431640625,
  'grad_norm': 0.019599013030529022,
  'learning_rate': 4.9091707553455694e-05,
  'epoch': 0.1817190623296384,
  'step': 3000},
 {'loss': 0.04711056518554688,
  'grad_norm': 0

In [11]:
# Save model and tokenizer
tokenizer.save_pretrained("./BERTmodel")
trainer.save_model("./BERTmodel")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]


In [12]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer
from scipy.special import softmax
from sklearn.metrics import roc_auc_score

# Load saved model
model = AutoModelForSequenceClassification.from_pretrained("./BERTmodel")
tokenizer = AutoTokenizer.from_pretrained("./BERTmodel")

# Create trainer
trainer = Trainer(model=model)

# Predict on test dataset
predictions = trainer.predict(test)

logits = predictions.predictions
y_true = predictions.label_ids

# Convert logits to probabilities
probs = softmax(logits, axis=1)
y_scores = probs[:, 1]  # probability of positive class

# ROC-AUC
auc = roc_auc_score(y_true, y_scores)

print(f"ROC-AUC: {auc:.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 970.74it/s]


ROC-AUC: 0.9661
